1. Đọc data từ HDFS

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Pakistan_Ecommerce_BigData") \
    .getOrCreate()

In [2]:
hdfs_path = "hdfs://localhost:9000/user/hadoop/ecommerce/Pakistan_Ecommerce.csv"

df = spark.read.csv(
    hdfs_path,
    header=True,
    inferSchema=True
)

df.show(5)

+-------+--------------+----------+--------------------+------+-----------+-----------+------------+-----------------+---------------------+---------------+--------------+------------+---------+-------+----+-----+--------------+------+----+-----------+----+----+----+----+----+
|item_id|        status|created_at|                 sku| price|qty_ordered|grand_total|increment_id|  category_name_1|sales_commission_code|discount_amount|payment_method|Working Date|BI Status|    MV |Year|Month|Customer Since|   M-Y|  FY|Customer ID|_c21|_c22|_c23|_c24|_c25|
+-------+--------------+----------+--------------------+------+-----------+-----------+------------+-----------------+---------------------+---------------+--------------+------------+---------+-------+----+-----+--------------+------+----+-----------+----+----+----+----+----+
| 211131|      complete|  7/1/2016|   kreations_YI 06-L|1950.0|          1|       1950|   100147443|  Women's Fashion|                   \N|              0|          

2. Kiểm tra số dòng, số cột ban đầu

In [3]:
print("Số dòng ban đầu:", df.count())
print("Số cột ban đầu:", len(df.columns))
print(df.columns)

Số dòng ban đầu: 1048586
Số cột ban đầu: 26
['item_id', 'status', 'created_at', 'sku', 'price', 'qty_ordered', 'grand_total', 'increment_id', 'category_name_1', 'sales_commission_code', 'discount_amount', 'payment_method', 'Working Date', 'BI Status', ' MV ', 'Year', 'Month', 'Customer Since', 'M-Y', 'FY', 'Customer ID', '_c21', '_c22', '_c23', '_c24', '_c25']


3. Đổi tên cột cho dễ xử lý

Dataset này có nhiều cột có khoảng trắng như Customer ID, Working Date, BI Status, nên nên đổi về dạng dễ dùng.

In [4]:
import re

def clean_column_name(name):
    name = name.strip()
    name = name.lower()
    name = re.sub(r"[^a-zA-Z0-9]+", "_", name)
    name = re.sub(r"_+", "_", name)
    name = name.strip("_")
    return name

new_columns = [clean_column_name(c) for c in df.columns]

df_clean = df.toDF(*new_columns)

df_clean.printSchema()
df_clean.show(5)

root
 |-- item_id: string (nullable = true)
 |-- status: string (nullable = true)
 |-- created_at: string (nullable = true)
 |-- sku: string (nullable = true)
 |-- price: double (nullable = true)
 |-- qty_ordered: string (nullable = true)
 |-- grand_total: string (nullable = true)
 |-- increment_id: string (nullable = true)
 |-- category_name_1: string (nullable = true)
 |-- sales_commission_code: string (nullable = true)
 |-- discount_amount: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- working_date: string (nullable = true)
 |-- bi_status: string (nullable = true)
 |-- mv: string (nullable = true)
 |-- year: string (nullable = true)
 |-- month: string (nullable = true)
 |-- customer_since: string (nullable = true)
 |-- m_y: string (nullable = true)
 |-- fy: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- c21: string (nullable = true)
 |-- c22: string (nullable = true)
 |-- c23: string (nullable = true)
 |-- c24: string (nullable

4. Xóa các cột rác

In [5]:
cols_to_drop = ["c21", "c22", "c23", "c24", "c25"]

cols_to_drop = [c for c in cols_to_drop if c in df_clean.columns]

df_clean = df_clean.drop(*cols_to_drop)

print("Các cột đã xóa:", cols_to_drop)
print("Số cột sau khi xóa:", len(df_clean.columns))

Các cột đã xóa: ['c21', 'c22', 'c23', 'c24', 'c25']
Số cột sau khi xóa: 21


5. Xóa các dòng trắng ở cuối file

In [6]:
core_cols = [
    "item_id",
    "created_at",
    "sku",
    "price",
    "qty_ordered",
    "grand_total"
]

core_cols = [c for c in core_cols if c in df_clean.columns]

before = df_clean.count()

df_clean = df_clean.dropna(how="all", subset=core_cols)

after = df_clean.count()

print("Số dòng trước khi xóa dòng trắng:", before)
print("Số dòng sau khi xóa dòng trắng:", after)
print("Số dòng trắng đã xóa:", before - after)

Số dòng trước khi xóa dòng trắng: 1048586
Số dòng sau khi xóa dòng trắng: 584535
Số dòng trắng đã xóa: 464051


6. Kiểm tra giá trị thiếu

In [7]:
from pyspark.sql.functions import col, sum as spark_sum, when

missing_df = df_clean.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df_clean.columns
])

missing_df.show(truncate=False)

+-------+------+----------+---+-----+-----------+-----------+------------+---------------+---------------------+---------------+--------------+------------+---------+---+----+-----+--------------+---+---+-----------+
|item_id|status|created_at|sku|price|qty_ordered|grand_total|increment_id|category_name_1|sales_commission_code|discount_amount|payment_method|working_date|bi_status|mv |year|month|customer_since|m_y|fy |customer_id|
+-------+------+----------+---+-----+-----------+-----------+------------+---------------+---------------------+---------------+--------------+------------+---------+---+----+-----+--------------+---+---+-----------+
|0      |15    |0         |20 |11   |11         |11         |11          |175            |137182               |11             |11            |11          |11       |11 |11  |11   |11            |22 |22 |22         |
+-------+------+----------+---+-----+-----------+-----------+------------+---------------+---------------------+---------------+----

7. Xử lý tiếp các dòng còn thiếu ở cột quan trọng

In [8]:
important_cols = [
    "item_id",
    "status",
    "created_at",
    "sku",
    "price",
    "qty_ordered",
    "grand_total",
    "category_name_1",
    "payment_method"
]

important_cols = [c for c in important_cols if c in df_clean.columns]

before = df_clean.count()

df_clean = df_clean.dropna(subset=important_cols)

after = df_clean.count()

print("Số dòng trước:", before)
print("Số dòng sau:", after)
print("Số dòng bị xóa:", before - after)

Số dòng trước: 584535
Số dòng sau: 584325
Số dòng bị xóa: 210


8. Kiểm tra lại số dòng cuối

In [9]:
print("Số dòng sau làm sạch:", df_clean.count())
print("Số cột sau làm sạch:", len(df_clean.columns))

df_clean.show(5)
df_clean.printSchema()

Số dòng sau làm sạch: 584325
Số cột sau làm sạch: 21
+-------+--------------+----------+--------------------+------+-----------+-----------+------------+-----------------+---------------------+---------------+--------------+------------+---------+-------+----+-----+--------------+------+----+-----------+
|item_id|        status|created_at|                 sku| price|qty_ordered|grand_total|increment_id|  category_name_1|sales_commission_code|discount_amount|payment_method|working_date|bi_status|     mv|year|month|customer_since|   m_y|  fy|customer_id|
+-------+--------------+----------+--------------------+------+-----------+-----------+------------+-----------------+---------------------+---------------+--------------+------------+---------+-------+----+-----+--------------+------+----+-----------+
| 211131|      complete|  7/1/2016|   kreations_YI 06-L|1950.0|          1|       1950|   100147443|  Women's Fashion|                   \N|              0|           cod|    7/1/2016|    

7. Ép kiểu dữ liệu số

In [10]:
from pyspark.sql.functions import regexp_replace

numeric_cols = ["price", "qty_ordered", "grand_total", "discount_amount", "mv"]

for c in numeric_cols:
    if c in df_clean.columns:
        df_clean = df_clean.withColumn(
            c,
            regexp_replace(col(c).cast("string"), ",", "").cast("double")
        )

df_clean.printSchema()

root
 |-- item_id: string (nullable = true)
 |-- status: string (nullable = true)
 |-- created_at: string (nullable = true)
 |-- sku: string (nullable = true)
 |-- price: double (nullable = true)
 |-- qty_ordered: double (nullable = true)
 |-- grand_total: double (nullable = true)
 |-- increment_id: string (nullable = true)
 |-- category_name_1: string (nullable = true)
 |-- sales_commission_code: string (nullable = true)
 |-- discount_amount: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- working_date: string (nullable = true)
 |-- bi_status: string (nullable = true)
 |-- mv: double (nullable = true)
 |-- year: string (nullable = true)
 |-- month: string (nullable = true)
 |-- customer_since: string (nullable = true)
 |-- m_y: string (nullable = true)
 |-- fy: string (nullable = true)
 |-- customer_id: string (nullable = true)



8. Làm sạch ngày tháng

In [11]:
from pyspark.sql.functions import col, trim, regexp_extract, concat_ws, when

# Dữ liệu hiện tại đang dạng 2016-07-01
df_clean = df_clean.withColumn(
    "created_at_raw",
    trim(col("created_at").cast("string"))
)

pattern = r"^(\d{4})-(\d{2})-(\d{2})"

df_clean = df_clean.withColumn(
    "order_year",
    regexp_extract(col("created_at_raw"), pattern, 1).cast("int")
)

df_clean = df_clean.withColumn(
    "order_month",
    regexp_extract(col("created_at_raw"), pattern, 2).cast("int")
)

df_clean = df_clean.withColumn(
    "order_day",
    regexp_extract(col("created_at_raw"), pattern, 3).cast("int")
)

df_clean = df_clean.withColumn(
    "created_date",
    when(
        col("order_year").isNotNull(),
        concat_ws(
            "-",
            col("order_year"),
            regexp_extract(col("created_at_raw"), pattern, 2),
            regexp_extract(col("created_at_raw"), pattern, 3)
        )
    ).otherwise(None)
)

df_clean.select(
    "created_at_raw",
    "created_date",
    "order_year",
    "order_month",
    "order_day"
).show(20, truncate=False)

+--------------+------------+----------+-----------+---------+
|created_at_raw|created_date|order_year|order_month|order_day|
+--------------+------------+----------+-----------+---------+
|7/1/2016      |null        |null      |null       |null     |
|7/1/2016      |null        |null      |null       |null     |
|7/1/2016      |null        |null      |null       |null     |
|7/1/2016      |null        |null      |null       |null     |
|7/1/2016      |null        |null      |null       |null     |
|7/1/2016      |null        |null      |null       |null     |
|7/1/2016      |null        |null      |null       |null     |
|7/1/2016      |null        |null      |null       |null     |
|7/1/2016      |null        |null      |null       |null     |
|7/1/2016      |null        |null      |null       |null     |
|7/1/2016      |null        |null      |null       |null     |
|7/1/2016      |null        |null      |null       |null     |
|7/1/2016      |null        |null      |null       |nul

9. Loại bỏ dòng giá trị không hợp lệ

In [12]:
if "price" in df_clean.columns:
    df_clean = df_clean.filter(col("price") >= 0)

if "qty_ordered" in df_clean.columns:
    df_clean = df_clean.filter(col("qty_ordered") > 0)

if "grand_total" in df_clean.columns:
    df_clean = df_clean.filter(col("grand_total") >= 0)

print("Số dòng sau khi lọc giá trị không hợp lệ:", df_clean.count())

Số dòng sau khi lọc giá trị không hợp lệ: 584238


10. Chuẩn hóa chữ trong các cột phân loại

In [13]:
from pyspark.sql.functions import lower, trim

text_cols = ["status", "category_name_1", "payment_method", "bi_status"]

for c in text_cols:
    if c in df_clean.columns:
        df_clean = df_clean.withColumn(c, lower(trim(col(c))))

df_clean.select([c for c in text_cols if c in df_clean.columns]).show(10)

+--------------+-----------------+--------------+---------+
|        status|  category_name_1|payment_method|bi_status|
+--------------+-----------------+--------------+---------+
|      complete|  women's fashion|           cod|    #ref!|
|      canceled|beauty & grooming|           cod|    gross|
|      canceled|  women's fashion|           cod|    gross|
|      complete|beauty & grooming|           cod|      net|
|order_refunded|          soghaat|           cod|    valid|
|      canceled|          soghaat|           cod|    gross|
|      complete|beauty & grooming|           cod|      net|
|      complete|          soghaat|           cod|      net|
|      canceled|mobiles & tablets| ublcreditcard|    gross|
|      canceled|mobiles & tablets|     mygateway|    gross|
+--------------+-----------------+--------------+---------+
only showing top 10 rows



11. Xóa dòng trùng lặp

In [14]:
before = df_clean.count()

df_clean = df_clean.dropDuplicates()

after = df_clean.count()

print("Số dòng trước khi xóa trùng:", before)
print("Số dòng sau khi xóa trùng:", after)
print("Số dòng bị xóa:", before - after)

Số dòng trước khi xóa trùng: 584238
Số dòng sau khi xóa trùng: 584238
Số dòng bị xóa: 0


12. Tạo thêm cột phục vụ phân tích

In [15]:
from pyspark.sql.functions import year, month, dayofmonth

if "created_at" in df_clean.columns:
    df_clean = df_clean.withColumn("order_year", year(col("created_at")))
    df_clean = df_clean.withColumn("order_month", month(col("created_at")))
    df_clean = df_clean.withColumn("order_day", dayofmonth(col("created_at")))

df_clean.show(5)

+-------+--------------+----------+--------------------+------+-----------+-----------+------------+-----------------+---------------------+---------------+--------------+------------+---------+------+----+-----+--------------+------+----+-----------+--------------+----------+-----------+---------+------------+
|item_id|        status|created_at|                 sku| price|qty_ordered|grand_total|increment_id|  category_name_1|sales_commission_code|discount_amount|payment_method|working_date|bi_status|    mv|year|month|customer_since|   m_y|  fy|customer_id|created_at_raw|order_year|order_month|order_day|created_date|
+-------+--------------+----------+--------------------+------+-----------+-----------+------------+-----------------+---------------------+---------------+--------------+------------+---------+------+----+-----+--------------+------+----+-----------+--------------+----------+-----------+---------+------------+
| 211415|order_refunded|  7/1/2016|      Audionic_B-710|1350.

In [16]:
helper_cols = [
    "created_at_raw",
    "working_date_raw",
    "customer_since_raw",
    "created_date"
]

helper_cols = [c for c in helper_cols if c in df_clean.columns]

df_clean = df_clean.drop(*helper_cols)

print("Đã xóa cột phụ:", helper_cols)
print("Số cột sau khi xóa cột phụ:", len(df_clean.columns))

Đã xóa cột phụ: ['created_at_raw', 'created_date']
Số cột sau khi xóa cột phụ: 24


13. Kiểm tra data sau khi làm sạch

In [17]:
print("Số dòng sau làm sạch:", df_clean.count())
print("Số cột sau làm sạch:", len(df_clean.columns))

df_clean.show(10)

Số dòng sau làm sạch: 584238
Số cột sau làm sạch: 24
+-------+--------------+----------+--------------------+------+-----------+-----------+------------+-----------------+---------------------+---------------+--------------+------------+---------+------+----+-----+--------------+------+----+-----------+----------+-----------+---------+
|item_id|        status|created_at|                 sku| price|qty_ordered|grand_total|increment_id|  category_name_1|sales_commission_code|discount_amount|payment_method|working_date|bi_status|    mv|year|month|customer_since|   m_y|  fy|customer_id|order_year|order_month|order_day|
+-------+--------------+----------+--------------------+------+-----------+-----------+------------+-----------------+---------------------+---------------+--------------+------------+---------+------+----+-----+--------------+------+----+-----------+----------+-----------+---------+
| 211415|order_refunded|  7/1/2016|      Audionic_B-710|1350.0|        1.0|     1350.0|   10

14. Lưu dữ liệu sạch lại lên HDFS

In [18]:
output_path = "hdfs://localhost:9000/ecom/ecom_clean_csv"

df_clean.write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv(output_path)

print("Đã lưu dữ liệu sạch dạng CSV vào:", output_path)

Đã lưu dữ liệu sạch dạng CSV vào: hdfs://localhost:9000/ecom/ecom_clean_csv


In [19]:
hdfs_path = "hdfs://localhost:9000/ecom/ecom_clean_csv"

df = spark.read.csv(
    hdfs_path,
    header=True,
    inferSchema=True
)

df.show(5)

+-------+--------+----------+--------------------+------+-----------+-----------+------------+---------------+---------------------+---------------+--------------+------------+---------+------+----+-----+--------------+------+----+-----------+----------+-----------+---------+
|item_id|  status|created_at|                 sku| price|qty_ordered|grand_total|increment_id|category_name_1|sales_commission_code|discount_amount|payment_method|working_date|bi_status|    mv|year|month|customer_since|   m_y|  fy|customer_id|order_year|order_month|order_day|
+-------+--------+----------+--------------------+------+-----------+-----------+------------+---------------+---------------------+---------------+--------------+------------+---------+------+----+-----+--------------+------+----+-----------+----------+-----------+---------+
| 211293|canceled|  7/1/2016|darzee_DP-234-B-P...|1450.0|        1.0|     1450.0|   100147554|  men's fashion|                   \N|            0.0| ublcreditcard|    7/

### 14. Lưu dữ liệu đã làm sạch xuống HDFS
Lưu dưới dạng Parquet để giữ nguyên kiểu dữ liệu (Schema) và tiết kiệm dung lượng.

In [20]:
output_path = "hdfs://localhost:9000/user/hadoop/ecommerce/Pakistan_Ecommerce_Clean.parquet"

# Lưu dữ liệu (ghi đè nếu đã tồn tại)
df_clean.write.mode("overwrite").parquet(output_path)

print(f"Đã lưu dữ liệu làm sạch tại: {output_path}")

Đã lưu dữ liệu làm sạch tại: hdfs://localhost:9000/user/hadoop/ecommerce/Pakistan_Ecommerce_Clean.parquet
